# A2e -- seed marker vs granule content

This is the fourth notebook of analysis **A2** (`plans/Round2_response_analysis_plan.md`). A2a, A2b
and A2c answer Reviewer #2's major point 6 from three directions; this notebook produces the two
results the response still needs and nothing else.

> *"each granule essentially expresses just the single marker it was detected on"*

**Part 1 -- per sample.** For MERSCOPE WT, MERSCOPE AD and Xenium 5K separately: the fraction of
granules carrying at least three unique genes, and, among granules assigned to a *pure* subtype
(`pre-syn`, `post-syn`, `dendrites`), the fraction carrying at least one further marker of that same
category beyond the seed marker itself.

**Part 2 -- WT + AD combined.** Whether granule content is randomly distributed or specifically
associated with the seed's functional category: group granules by the category of their seeding
marker, delete each granule's own seed gene's counts, tabulate the remaining subtype-marker
transcripts by category, and test the resulting table with an omnibus chi-square plus a Fisher's
exact test on each diagonal cell.

**Self-contained.** It reads only published artifacts -- the granule x gene AnnData and the subtype
label parquets. No detection is rerun, `transcripts.parquet` is never opened, and it does not depend
on A2a, A2b or A2c having been run.

Everything lands in `output/a2e/`.

## 0. Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.special import gammaln
from scipy.stats import chi2, chi2_contingency, fisher_exact, hypergeom

sys.path.insert(0, str(Path.cwd()))          # run from R2_revision/sparsity_structure/
import a2_config as C
import a2_common as A2

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

# -------------------- settings -------------------- #
# There are no run gates: this notebook is a single unified pass. The Xenium arm is skipped
# automatically if its outputs are absent (see the loading cell), and the correctness gates in
# section 5 always run -- they cost a 1,000-granule probe and a small CSV read, and there is no
# reason for correctness to be opt-in.
N_PERM = 2000              # granule-label shuffles in section 4  # matches C.GROUP_TEST_N_PERM
PERM_SEED = 0                                                     # matches C.GROUP_TEST_SEED

C.ensure_dirs()
OUT = C.A2E_DIR
T0 = time.time()
print("writing to", OUT, "| N_PERM =", N_PERM)

### Loading

Two provenance traps here, both real.

**`granule_id` is not a key in the combined object.** It restarts at `gnl_0` for the AD half, so it
has only 681,337 distinct values across 1,080,146 rows. The subtype parquet is written in the same
row order as the h5ad, so this aligns positionally and *asserts* the `(sample, granule_id)` pair
matches rather than merging on a column that would silently mismatch.

**`obs["gene"]` is the seed marker, and it is one marker.** `merge_sphere()` folds overlapping
spheres seeded by different markers and keeps exactly one of the two `gene` values
(`model.py:323-377`, either row A's or a wholesale copy of row B's) -- the other is discarded. That
is the confound section 4's second arm exists to control.

In [ ]:
def _load_labels(path, obs, sample_col=None):
    """Subtype labels, aligned positionally with an assertion instead of a merge."""
    lab = pd.read_parquet(path)
    assert len(lab) == obs.shape[0], (len(lab), obs.shape[0])
    assert (lab["granule_id"].astype(str).to_numpy()
            == obs["granule_id"].astype(str).to_numpy()).all(), "granule_id order differs"
    if sample_col is not None:
        assert (lab["sample"].astype(str).to_numpy()
                == obs[sample_col].astype(str).to_numpy()).all(), "sample order differs"
    return lab["granule_subtype_manual_simple"].astype(str).to_numpy()


arms = {}

# ---- MERSCOPE pair 1, the published combined WT+AD object --------------------------------- #
mer = sc.read_h5ad(C.COMBINED_GRANULE_ADATA)
mer_subtype = _load_labels(C.COMBINED_SUBTYPE_LABELS, mer.obs, sample_col="batch")
mer_genes = list(mer.var_names)
mer_nc = [g for g in pd.read_csv(C.nc_path("WT"))["Gene"] if g in set(mer_genes)]
mer_cat = C.marker_category_map(mer_genes)
mer_batch = mer.obs["batch"].astype(str).to_numpy()
mer_seed = mer.obs["gene"].astype(str).to_numpy()

for s in C.SAMPLES:
    m = mer_batch == C.dataset(s)
    arms[s] = dict(platform="MERSCOPE", counts=mer.layers["counts"][m], genes=mer_genes,
                   seed=mer_seed[m], subtype=mer_subtype[m], cat_map=mer_cat,
                   seed_genes=list(C.SYN_GENES), nc=mer_nc)

# Part 2 runs on WT and AD together, as one object.
combined = dict(platform="MERSCOPE", counts=mer.layers["counts"], genes=mer_genes,
                seed=mer_seed, subtype=mer_subtype, cat_map=mer_cat,
                seed_genes=list(C.SYN_GENES), nc=mer_nc, sample=mer_batch)

# ---- Xenium 5K ----------------------------------------------------------------------------- #
# A single sample, a 5006-gene panel, and its OWN 16-marker seed list -- not a subset of the
# MERSCOPE 20. Only granule_adata_tsne.h5ad keeps per-granule seed identity; the *_merged_genes*
# files in the same directory carry gene == "merged" on every row.
XENIUM_AVAILABLE = C.XENIUM_GRANULE_ADATA.exists() and C.XENIUM_SUBTYPE_LABELS.exists()
if XENIUM_AVAILABLE:
    xen = sc.read_h5ad(C.XENIUM_GRANULE_ADATA)
    xen_genes = list(xen.var_names)
    arms["Xenium"] = dict(
        platform="Xenium", counts=xen.layers["counts"], genes=xen_genes,
        seed=xen.obs["gene"].astype(str).to_numpy(),
        subtype=_load_labels(C.XENIUM_SUBTYPE_LABELS, xen.obs),
        cat_map=C.marker_category_map(xen_genes),
        seed_genes=list(C.XENIUM_SYN_GENES),
        nc=[g for g in pd.read_csv(C.XENIUM_NC)["Gene"] if g in set(xen_genes)])
else:
    print(f"[skip] Xenium arm: {C.XENIUM_GRANULE_ADATA} not present. Sections 1-3 run on WT and AD\n"
          f"       only; section 4 is MERSCOPE-only by design and is unaffected.")

for name, a in arms.items():
    print(f"{name:8s} {a['counts'].shape[0]:>9,} granules x {len(a['genes']):>5,} genes "
          f"| {len(a['cat_map']):>2d} subtype markers | {len(a['nc']):>3d} negative controls")
print(f"combined {combined['counts'].shape[0]:>9,} granules")

## 1. Marker inventory

Written **before** any statistic, so that every denominator used downstream is on the record.

The "34 subtype marker list" is `a2_config.REF_GENES`, categorised by `a2_config.MARKER_GENES`
(both copied verbatim from `code/benchmark/benchmark_subtyping.ipynb` cell 4). `MARKER_GENES` has 38
entries over 37 unique genes -- `Dlg4` is listed under both post-syn and dendrites --  and
`REF_GENES` is that set intersected with the 290-gene MERSCOPE panel, `Snap25`, `Actb` and `Sptnb4`
being off it. `C.marker_category_map()` resolves `Dlg4` to post-syn, which is where `REF_GENES`
column order groups it too, so this reproduces the published subtype heatmap's grouping rather than
inventing a convention.

**Xenium is a reduced panel and this table is the reason its numbers must never be quoted as
comparable to MERSCOPE's.** Eleven of the 34 are off-panel there (`Nrxn1, Syt1, Cplx2, Homer2,
Nlgn1, Shank1, Ddn, Map1a, Ank3, Nav1, Nfasc`) and `Snap25` is on it, giving 24 markers -- with only
**two** dendritic and **two** axonal.

Writes `marker_inventory.csv`.

In [ ]:
rows = []
for name, a in arms.items():
    seeds = set(a["seed_genes"])
    for cat in C.CONTENT_CATEGORIES:
        members = [g for g, c in a["cat_map"].items() if c == cat]
        seed_m = [g for g in members if g in seeds]
        nonseed_m = [g for g in members if g not in seeds]
        rows.append(dict(
            sample=name, platform=a["platform"], category=cat,
            n_markers=len(members), n_seed_markers=len(seed_m), n_nonseed_markers=len(nonseed_m),
            markers=";".join(members), seed_markers=";".join(seed_m),
            nonseed_markers=";".join(nonseed_m)))

inventory = pd.DataFrame(rows)
inventory.to_csv(OUT / "marker_inventory.csv", index=False)

print(inventory.pivot_table(index="sample", columns="category",
                            values=["n_markers", "n_seed_markers", "n_nonseed_markers"],
                            aggfunc="sum").to_string())
print("\nmarkers per sample:",
      inventory.groupby("sample")["n_markers"].sum().to_dict())

## 2. Granule complexity per sample

`A2.unique_gene_counts` is the same primitive A2a uses -- `n_genes = (counts > 0).sum(axis=1)` on
`layers["counts"]`, the raw matrix -- so the per-sample `frac_ge_3` under the `nonNC` convention
must reproduce `output/a2a/multigene/retention_by_region.csv` exactly. Section 5 asserts it.

Neither panel has blank probes: the negative controls are real nuclear-enriched genes, 19 of them on
MERSCOPE and 293 on Xenium. "Unique genes" therefore needs a stated denominator, and both are
exported rather than one being chosen silently. `nonNC` is the headline
(`C.EXCLUDE_NC_FROM_COMPLEXITY = True`); the 293-gene MERSCOPE/Xenium asymmetry is exactly why.

Writes `complexity_by_sample.csv`.

In [ ]:
rows = []
for name, a in arms.items():
    for convention, exclude in (("nonNC", a["nc"]), ("all_panel", None)):
        n_reads, n_genes = A2.unique_gene_counts(a["counts"], a["genes"], exclude_genes=exclude)
        row = dict(
            sample=name, platform=a["platform"], counting=convention,
            n_panel_genes=len(a["genes"]),
            n_excluded_nc=0 if exclude is None else len(exclude),
            n_granules=int(n_genes.size),
            mean_n_genes=float(n_genes.mean()), median_n_genes=float(np.median(n_genes)),
            mean_n_reads=float(n_reads.mean()), median_n_reads=float(np.median(n_reads)))
        for k in C.A2E_COMPLEXITY_LEVELS:
            row[f"n_ge_{k}"] = int((n_genes >= k).sum())
            row[f"frac_ge_{k}"] = float((n_genes >= k).mean())
        rows.append(row)

complexity = pd.DataFrame(rows)
complexity.to_csv(OUT / "complexity_by_sample.csv", index=False)

print(complexity[["sample", "counting", "n_granules", "median_n_genes",
                  "n_ge_3", "frac_ge_3"]].to_string(index=False))

## 3. Same-category content in pure-subtype granules

Restricted to granules whose published label is one of the three **pure** subtypes. There is no
fourth: `axons` maps to `[]` in every published cluster -> subtype mapping
(`benchmark_subtyping.ipynb` cell 21, `other_analysis/Xenium_5K/2_subtyping.ipynb` cell 10) and no
axon-containing label exists in either parquet. Xenium recovers axonal signal only as a mixed
`pre & axon` cluster. That is a fact about the K-means clusters and constrains nothing in section 4,
whose axis is the seed marker.

For a granule labelled `c`, the question is whether it carries a marker of `c` **other than its own
seed gene**. `frac_seed_in_own_category` is reported beside the answer because that qualifier only
bites when the seed is itself in `c` -- a granule labelled `pre-syn` can perfectly well have been
seeded by a post-syn marker. It is a diagnostic, not a filter: the headline fraction stays on the
full pure-subtype population.

Read the Xenium rows against `marker_inventory.csv`. Its `dendrites` category has two markers
(`Cyfip2`, `Map2`), both of which are also seeds, so "another dendritic marker beyond the seed"
there has exactly one candidate gene.

Writes `same_category_content.csv`.

In [ ]:
rows = []
for name, a in arms.items():
    gidx = {g: i for i, g in enumerate(a["genes"])}
    for c in C.PURE_SUBTYPES:
        mask = a["subtype"] == c
        if not mask.any():
            print(f"[skip] {name}: no granules labelled {c}")
            continue
        markers = [g for g, cc in a["cat_map"].items() if cc == c]
        seed_sub = a["seed"][mask]

        # <= 13 marker columns, so densifying the submatrix is cheap and keeps the seed-column
        # bookkeeping readable.
        dense = a["counts"][mask][:, [gidx[g] for g in markers]].toarray() > 0
        n_present = dense.sum(axis=1)

        own = np.zeros(seed_sub.shape[0], dtype=bool)
        for j, g in enumerate(markers):
            hit = seed_sub == g
            if hit.any():
                own[hit] = dense[hit, j]
        n_other = n_present - own.astype(int)

        rows.append(dict(
            sample=name, platform=a["platform"], subtype=c,
            n_granules=int(mask.sum()),
            n_markers_in_category=len(markers), markers=";".join(markers),
            n_with_other_same_category=int((n_other >= 1).sum()),
            frac_with_other_same_category=float((n_other >= 1).mean()),
            frac_seed_in_own_category=float(np.isin(seed_sub, markers).mean()),
            median_n_same_category_markers=float(np.median(n_present)),
            median_n_other_same_category=float(np.median(n_other))))

same_category = pd.DataFrame(rows)
same_category.to_csv(OUT / "same_category_content.csv", index=False)

print(same_category[["sample", "subtype", "n_granules", "n_markers_in_category",
                     "frac_with_other_same_category", "frac_seed_in_own_category"]]
      .to_string(index=False))

## 4. Seed category vs co-detected content

MERSCOPE WT + AD combined, all granules. The test as specified:

1. group granules by the category of their seeding marker (`obs["gene"]` -> `C.marker_category_map`)
2. remove each granule's own seed gene's counts, leaving its remaining subtype-marker transcripts
3. sum those within each seed-category group and classify them by the same four categories,
   giving a 4 x 4 table of seed category x co-detected content category
4. omnibus chi-square of independence on the table, then a Fisher's exact test on each diagonal cell

The table is 4 x 4 rather than 3 x 3 because the *seed* axis has all four categories -- MERSCOPE
seeds on `Mapt`, `Nav1`, `Nfasc` and `Tubb3`. The 3 x 3 axon-dropped submatrix is reported beside
it so either framing can be quoted.

### Two arms, because merging manufactures part of the diagonal

`merge_sphere()` keeps one seed marker of a merged pair and discards the other, and 8 of the 20
MERSCOPE seeds are pre-syn. A pre-syn-seeded granule is therefore likelier to contain a second
pre-syn marker *by construction*, independently of any biology.

| arm | content tally |
|---|---|
| `all_content` | the analysis as specified -- remove only the granule's own seed gene |
| `nonseed_content` | remove **every** seed marker, leaving only the 14 non-seed subtype markers |

`nonseed_content` is the arm that cannot be attributed to merging, and it is the one to quote when
the confound is raised. It is also thin: per `marker_inventory.csv` it rests on 4 pre-syn, 8
post-syn, **1** dendritic (`Map2`) and **1** axonal (`Ank3`) marker.

### Two inferences, reported side by side

The chi-square's unit is the **transcript**, and transcripts within a granule are not independent.
That is a real objection, so both inferences are computed and neither replaces the other:

| columns | what they are |
|---|---|
| `chi2`, `dof`, `p`, `neg_log10_p`, `cramers_v` | the asymptotic test on the transcript table, dependence and all -- the analysis exactly as specified |
| `chi2_perm_mean`, `design_effect`, `p_perm`, `n_perm` | 2,000 shuffles of the seed-category label **across granules** |

The permutation is the honest null: each granule's whole content vector travels with it, so every
bit of within-granule dependence is preserved and the granule -- which *is* independent -- is the
exchangeable unit. `design_effect = mean(chi2_perm) / dof` is then the overdispersion measured
rather than assumed; at 1.0 the asymptotic test needs no correction at all.

All chi-square tests here are computed without Yates' continuity correction, so the 2 x 2
compartment table in section 4c is the same statistic as the larger tables rather than a corrected
one; at these counts the difference is under one part in 10,000 either way.

`p_perm` cannot go below `1 / (N_PERM + 1)` = 5.0e-4, so read it as a floor, not as an estimate of
how small the p-value is; `neg_log10_p` beside it carries that. Both `p` columns underflow to `0.0`
at these counts, so `neg_log10_p` is computed in log space -- exactly for the Fisher tests
(`hypergeom.logsf`) and asymptotically for the omnibus chi-square.

### The diagonal is not the right frame, and the tests are two-sided because of it

Same-category association does not hold uniformly across the four labels, and the Fisher tests are
therefore **two-sided with an explicit `direction` column**. A one-sided `greater` test returns
p = 1 for a depleted cell, which would report the largest deviations as null.

The reason is anatomical, not statistical. These four marker sets are **two compartments split into
two overlapping labels each**: the postsynaptic density sits inside dendritic spines, and
presynaptic terminals are axonal structures. So `post-syn` and `dendrites` label one physical
compartment and cannot separate from each other, and neither can `pre-syn` and `axons`. Section 4c
collapses to the two compartments, which is where the question is actually testable.
`C.COMPARTMENT_OF` fixes that grouping a priori from standard neuroanatomy -- it is not a
regrouping chosen after seeing the table.

`fold_vs_expected` on all 16 cells makes the block structure visible, so the omnibus result has a
stated explanation rather than just a p-value.

Writes `seed_content_table.csv`, `seed_content_tests.csv`, `seed_content_diagonal.csv`.

In [ ]:
CAT = list(C.CONTENT_CATEGORIES)
COMP = list(C.COMPARTMENTS)
cat_map = combined["cat_map"]
gidx = {g: i for i, g in enumerate(combined["genes"])}
marker_genes = list(cat_map)                       # the 34, in REF_GENES order
seedset = set(combined["seed_genes"])

seed_cat = pd.Series(combined["seed"]).map(cat_map).to_numpy()
assert not pd.isna(seed_cat).any(), "a seed marker has no category"
seed_code = np.array([CAT.index(s) for s in seed_cat])          # int codes for np.bincount

# Dense 1.08M x 34 marker block: small enough to hold, and it makes the per-granule seed-column
# bookkeeping below explicit rather than clever.
M = combined["counts"][:, [gidx[g] for g in marker_genes]].toarray()
assert np.all(M == np.floor(M)), "layers['counts'] is not integral -- wrong layer?"
M = M.astype(np.int32)          # per-granule counts are tiny; int64 here costs 300 MB for nothing

# Per-granule content totals by category, before any removal.
cat_cols = {c: [j for j, g in enumerate(marker_genes) if cat_map[g] == c] for c in CAT}
base = np.column_stack([M[:, cat_cols[c]].sum(axis=1) for c in CAT])

content = {}

# arm 1: subtract each granule's own seed gene, from its seed's category column only
A_all = base.copy()
for j, g in enumerate(marker_genes):
    hit = combined["seed"] == g
    if hit.any():
        A_all[hit, CAT.index(cat_map[g])] -= M[hit, j]
content["all_content"] = A_all.astype(np.float64)          # float for np.bincount weights

# arm 2: subtract every seed marker, per category
A_ns = base.copy()
for k, c in enumerate(CAT):
    cols = [j for j in cat_cols[c] if marker_genes[j] in seedset]
    if cols:
        A_ns[:, k] -= M[:, cols].sum(axis=1)
content["nonseed_content"] = A_ns.astype(np.float64)

for arm, A in content.items():
    assert (A >= 0).all(), f"{arm}: negative content after seed removal"
    print(f"{arm:16s} {A.sum():>12,.0f} transcripts over {A.shape[0]:,} granules "
          f"| mean {A.sum(axis=1).mean():.2f} per granule")

In [ ]:
def _table(labels, A):
    """4 x 4 seed-category x content-category transcript table. Rows follow `labels`."""
    return np.column_stack([np.bincount(labels, weights=A[:, k], minlength=len(CAT))
                            for k in range(len(CAT))])


def _collapse(table):
    """4 x 4 -> 2 x 2 over C.COMPARTMENT_OF, both axes."""
    idx = {p: [i for i, c in enumerate(CAT) if C.COMPARTMENT_OF[c] == p] for p in COMP}
    return np.array([[table[np.ix_(idx[r], idx[c])].sum() for c in COMP] for r in COMP])


def _folds(table):
    """Diagonal fold enrichment: how much likelier content of category i is in granules seeded by
    i than in granules seeded by anything else. This is the quantity the response sentence quotes."""
    out = np.empty(table.shape[0])
    tot = table.sum()
    for i in range(table.shape[0]):
        a, row, col = table[i, i], table[i].sum(), table[:, i].sum()
        out[i] = (a / row) / ((col - a) / (tot - row))
    return out


def _neg_log10_sf(stat, dof):
    """-log10 of the chi-square upper tail, evaluated so that it survives underflow.

    scipy's chi2.logsf returns -inf once the tail underflows, which happens well before these
    statistics do. The fallback is the leading asymptotic term of the upper incomplete gamma,
    Q(k, x) ~ x^(k-1) e^-x / Gamma(k) -- accurate to far more digits than a p-value of this
    magnitude deserves. It exists so the number is reportable at all, not so it is quoted.
    """
    lsf = chi2.logsf(stat, dof)
    if not np.isfinite(lsf):
        x, k = stat / 2.0, dof / 2.0
        lsf = (k - 1) * np.log(x) - x - gammaln(k)
    return float(-lsf / np.log(10))


def _omnibus(table, label, arm):
    """Asymptotic chi-square of independence on the transcript table, with Cramer's V."""
    assert table.sum(axis=0).all() and table.sum(axis=1).all(), (
        f"{arm}/{label}: an all-zero row or column -- chi-square is undefined:\n{table}")
    # correction=False: chi2_contingency applies Yates' continuity correction automatically when
    # dof == 1, which would silently make only the 2x2 compartment table a different statistic from
    # the others. At these counts it shifts chi2 by <1 part in 10^4; turning it off keeps every
    # table in this notebook on the same footing and needs no footnote.
    stat, p, dof, expected = chi2_contingency(table, correction=False)
    n = table.sum()
    return dict(arm=arm, table=label, n_rows=table.shape[0], n_cols=table.shape[1],
                n_transcripts=int(n), chi2=float(stat), dof=int(dof), p=float(p),
                neg_log10_p=_neg_log10_sf(stat, dof),
                cramers_v=float(np.sqrt(stat / (n * (min(table.shape) - 1))))), expected


def _diagonal_2x2(table, i):
    """Collapse to (seed category i vs rest) x (content category i vs rest)."""
    a = int(table[i, i])
    b = int(table[i].sum() - a)
    c = int(table[:, i].sum() - a)
    return np.array([[a, b], [c, int(table.sum()) - a - b - c]], dtype=np.int64)


def _enrichment(tab2):
    """Two-sided Fisher's exact test on a collapsed 2 x 2, plus the fold and its direction.

    Two-sided is not optional here: a one-sided `greater` test returns p = 1 on a depleted cell,
    which would report the largest deviations in this table as null. `neg_log10_p` takes the
    matching hypergeometric tail in log space, so it stays exact where `p` underflows to 0.0.
    """
    (a, b), (c, d) = tab2
    n_total = int(a + b + c + d)
    frac_own = a / (a + b) if (a + b) else np.nan
    frac_other = c / (c + d) if (c + d) else np.nan
    fold = frac_own / frac_other if frac_other else np.inf
    odds, p = fisher_exact(tab2)
    N, K, n = n_total, int(a + c), int(a + b)
    lsf = hypergeom.logsf(int(a) - 1, N, K, n) if fold >= 1 else hypergeom.logcdf(int(a), N, K, n)
    return dict(n_own=int(a), n_in_seed_group=int(a + b), n_total=n_total,
                frac_own=float(frac_own), frac_other=float(frac_other),
                fold_enrichment=float(fold), direction="enriched" if fold >= 1 else "depleted",
                odds_ratio=float(odds), p=float(p),
                neg_log10_p=float(-lsf / np.log(10)) if np.isfinite(lsf) else np.inf)


def _perm_p(obs, null, log_scale=False):
    """(1 + #{null at least as extreme}) / (n + 1). Two-sided on the log scale for folds."""
    null = np.asarray(null, dtype=float)
    if log_scale:
        hit = np.abs(np.log(null)) >= np.abs(np.log(obs))
    else:
        hit = null >= obs
    return float((1 + hit.sum()) / (null.size + 1))

In [ ]:
# ---- 2,000 granule-label shuffles per arm ------------------------------------------------- #
# The whole content vector travels with its granule, so within-granule dependence is preserved
# exactly and the granule is the exchangeable unit. Records the omnibus chi2, the four diagonal
# folds and the compartment 2x2 fold, so every statistic in section 4 gets a permutation p.
null = {}
rng = np.random.default_rng(PERM_SEED)
for arm, A in content.items():
    t0 = time.time()
    perm = seed_code.copy()
    chi2s, folds4, folds2 = [], [], []
    for _ in range(N_PERM):
        rng.shuffle(perm)
        Tp = _table(perm, A)
        chi2s.append(chi2_contingency(Tp)[0])
        folds4.append(_folds(Tp))
        folds2.append(_folds(_collapse(Tp)))
    null[arm] = dict(chi2=np.array(chi2s), fold4=np.array(folds4), fold2=np.array(folds2))
    print(f"{arm:16s} {N_PERM} permutations in {time.time() - t0:5.1f}s "
          f"| null chi2 mean {np.mean(chi2s):6.2f}  max {np.max(chi2s):6.2f}")

In [ ]:
table_rows, test_rows, diag_rows, comp_rows = [], [], [], []
P_FLOOR = 1.0 / (N_PERM + 1)

for arm, A in content.items():
    table = _table(seed_code, A)
    n_gran = {s: int((seed_cat == s).sum()) for s in CAT}

    # -- omnibus, asymptotic + permutation ------------------------------------------------- #
    for label, idx in (("4x4", list(range(len(CAT)))),
                       ("3x3", [CAT.index(c) for c in C.PURE_SUBTYPES])):
        row, expected = _omnibus(table[np.ix_(idx, idx)], label, arm)
        row["n_granules"] = sum(n_gran[CAT[i]] for i in idx)
        if label == "4x4":
            exp_4x4 = expected
            row.update(n_perm=N_PERM, chi2_perm_mean=float(null[arm]["chi2"].mean()),
                       chi2_perm_max=float(null[arm]["chi2"].max()),
                       design_effect=float(null[arm]["chi2"].mean() / row["dof"]),
                       p_perm=_perm_p(row["chi2"], null[arm]["chi2"]), p_perm_floor=P_FLOOR)
        else:
            # The 3x3 is a reported alternative framing, not the headline, so it keeps the
            # asymptotic test only rather than spending a second permutation on it.
            row.update(n_perm=None, chi2_perm_mean=None, chi2_perm_max=None,
                       design_effect=None, p_perm=None, p_perm_floor=None)
        test_rows.append(row)

    # -- the 16 cells, with fold vs expected ----------------------------------------------- #
    obs_f4 = _folds(table)
    for i, s in enumerate(CAT):
        for j, cc in enumerate(CAT):
            table_rows.append(dict(arm=arm, seed_category=s, content_category=cc,
                                   n_transcripts=int(table[i, j]),
                                   expected=float(exp_4x4[i, j]),
                                   fold_vs_expected=float(table[i, j] / exp_4x4[i, j]),
                                   n_granules_in_seed_group=n_gran[s]))
        diag_rows.append(dict(
            arm=arm, category=s, n_granules_in_seed_group=n_gran[s],
            n_markers_in_category=len(cat_cols[s]),
            **_enrichment(_diagonal_2x2(table, i)),
            fold_null_mean=float(null[arm]["fold4"][:, i].mean()),
            fold_null_min=float(null[arm]["fold4"][:, i].min()),
            fold_null_max=float(null[arm]["fold4"][:, i].max()),
            p_perm=_perm_p(obs_f4[i], null[arm]["fold4"][:, i], log_scale=True),
            p_perm_floor=P_FLOOR, n_perm=N_PERM))

    # -- section 4c: the compartment collapse ---------------------------------------------- #
    T2 = _collapse(table)
    row2, exp2 = _omnibus(T2, "2x2", arm)
    obs_f2 = _folds(T2)
    for i, p_ in enumerate(COMP):
        # Diagonal cells get the same two-sided Fisher test as the 4x4 diagonals, so one Methods
        # sentence covers every diagonal in the analysis. The chi-square quantities keep their own
        # prefixed column names rather than being overwritten by the Fisher ones.
        fisher = _enrichment(_diagonal_2x2(T2, i))
        assert np.isclose(fisher["fold_enrichment"], obs_f2[i]), (
            f"{arm}/{p_}: Fisher and _folds disagree on the compartment fold "
            f"({fisher['fold_enrichment']} vs {obs_f2[i]})")
        for j, q_ in enumerate(COMP):
            row = dict(
                arm=arm, seed_compartment=p_, content_compartment=q_,
                n_transcripts=int(T2[i, j]), expected=float(exp2[i, j]),
                fold_vs_expected=float(T2[i, j] / exp2[i, j]),
                diagonal=(i == j),
                chi2=row2["chi2"], dof=row2["dof"], cramers_v=row2["cramers_v"],
                chi2_neg_log10_p=row2["neg_log10_p"])
            row.update(fisher if i == j else {k: np.nan for k in fisher})
            row.update(
                fold_null_mean=float(null[arm]["fold2"][:, i].mean()) if i == j else np.nan,
                fold_null_max=float(null[arm]["fold2"][:, i].max()) if i == j else np.nan,
                p_perm=_perm_p(obs_f2[i], null[arm]["fold2"][:, i], log_scale=True) if i == j
                       else np.nan,
                p_perm_floor=P_FLOOR, n_perm=N_PERM)
            comp_rows.append(row)
    a2 = (T2[0, 0] / T2[0].sum()) / (T2[1, 0] / T2[1].sum())
    print(f"{arm:16s} axonal-content fold in axonal- vs dendritic-seeded granules = {a2:.3f}")

seed_content_table = pd.DataFrame(table_rows)
seed_content_tests = pd.DataFrame(test_rows)
seed_content_diagonal = pd.DataFrame(diag_rows)
seed_content_compartment = pd.DataFrame(comp_rows)
seed_content_diagonal["q_bh"] = (seed_content_diagonal.groupby("arm", sort=False)["p"]
                                 .transform(lambda x: A2.bh_fdr(x.to_numpy())))

seed_content_table.to_csv(OUT / "seed_content_table.csv", index=False)
seed_content_tests.to_csv(OUT / "seed_content_tests.csv", index=False)
seed_content_diagonal.to_csv(OUT / "seed_content_diagonal.csv", index=False)
seed_content_compartment.to_csv(OUT / "seed_content_compartment.csv", index=False)

In [ ]:
for arm in content:
    t = seed_content_table.query("arm == @arm")
    print(f"\n=== {arm} ===  transcripts (rows = seed category)")
    print(t.pivot(index="seed_category", columns="content_category", values="n_transcripts")
           .reindex(index=CAT, columns=CAT).to_string())
    print("  fold vs expected")
    print(t.pivot(index="seed_category", columns="content_category", values="fold_vs_expected")
           .reindex(index=CAT, columns=CAT).round(2).to_string())

print("\n--- omnibus: asymptotic (transcript unit) beside the granule-label permutation ---")
print(seed_content_tests[["arm", "table", "n_transcripts", "chi2", "dof", "neg_log10_p",
                          "cramers_v", "chi2_perm_mean", "design_effect", "p_perm"]]
      .to_string(index=False))
print("\n--- diagonal cells: two-sided Fisher, with the permutation null band ---")
print(seed_content_diagonal[["arm", "category", "n_markers_in_category", "fold_enrichment",
                             "direction", "fold_null_min", "fold_null_max", "neg_log10_p",
                             "q_bh", "p_perm"]].to_string(index=False))
print("\n--- compartment collapse (a priori grouping, C.COMPARTMENT_OF) ---")
print(seed_content_compartment[["arm", "seed_compartment", "content_compartment",
                                "n_transcripts", "fold_vs_expected", "fold_enrichment",
                                "direction", "odds_ratio", "neg_log10_p", "cramers_v",
                                "p_perm"]].round(4).to_string(index=False))

### 4c. The compartment collapse -- where the question is actually testable

`seed_content_compartment.csv` is the frame in which "does content track the seed's category?" is
answerable. The four-way diagonal cannot answer it, because two of the four labels are sub-labels of
one physical compartment and one is a sub-label of the other: `post-syn` and `dendrites` both label
the somatodendritic compartment, `pre-syn` and `axons` both label the axonal one. A sub-label cannot
be shown to predict itself *in preference to* the compartment it lives inside.

Collapsed, the association is clean, and -- the point that matters -- **the same in both arms**, so
it is not an artifact of `merge_sphere` keeping one seed of a merged pair.

The two diagonal cells carry the same two-sided Fisher's exact test as the 4 x 4 diagonals
(`odds_ratio`, `p`, `neg_log10_p`, `direction`), so one Methods sentence covers every diagonal in
this notebook, alongside the chi-square (`chi2`, `chi2_neg_log10_p`, `cramers_v`) and the
permutation p. Note that the Fisher odds ratio and `fold_enrichment` answer different questions --
an odds ratio is not a ratio of proportions -- and must not be quoted interchangeably.

Two things to read off `nonseed_content` in particular. Its dendritic content is `Map2` alone, and
MAP2 is the canonical somatodendritic marker, actively excluded from axons; its fold vs expected in
axonally-seeded granules is the closest thing this panel has to a built-in positive control. And its
axonal diagonal rests on `Ank3` alone, so that cell carries a fold, not independent support.

`C.COMPARTMENT_OF` is fixed a priori from standard neuroanatomy and lives in `a2_config.py` beside
the marker sets, not here -- so that "specified before the table was seen" is checkable rather than
asserted.

### 4b. The same question with the granule as the unit

Section 4 counts transcripts and measures the resulting overdispersion; this counts granules, which
are independent by construction, so it needs no correction at all. For each category `c`, one clean
2 x 2 over the whole combined object:

|                       | carries a non-seed marker of `c` | does not |
|---|---|---|
| **seed category is `c`** | | |
| **seed category is not `c`** | | |

Content is restricted to the non-seed markers throughout, so this answers the merge confound and the
independence objection at once. It is the number to reach for if the transcript-level p-value is
challenged rather than the effect.

Writes `seed_content_granule_level.csv`.

In [ ]:
nonseed_cols = {c: [j for j in cat_cols[c] if marker_genes[j] not in seedset] for c in CAT}
present = {c: ((M[:, nonseed_cols[c]] > 0).any(axis=1) if nonseed_cols[c]
               else np.zeros(M.shape[0], dtype=bool))
           for c in CAT}

rows = []
for c in CAT:
    own, has = seed_cat == c, present[c]
    tab2 = np.array([[int((own & has).sum()), int((own & ~has).sum())],
                     [int((~own & has).sum()), int((~own & ~has).sum())]], dtype=np.int64)
    rows.append(dict(category=c, n_nonseed_markers=len(nonseed_cols[c]),
                     nonseed_markers=";".join(marker_genes[j] for j in nonseed_cols[c]),
                     n_seed_in_category=int(own.sum()), **_enrichment(tab2)))

granule_level = pd.DataFrame(rows)
granule_level["q_bh"] = A2.bh_fdr(granule_level["p"].to_numpy())
granule_level.to_csv(OUT / "seed_content_granule_level.csv", index=False)
print(granule_level[["category", "n_nonseed_markers", "n_seed_in_category", "frac_own",
                     "frac_other", "fold_enrichment", "direction", "neg_log10_p", "q_bh"]]
      .to_string(index=False))

## 5. Correctness gates

Cheap assertions on everything the two results depend on. They always run.

In [ ]:
from collections import Counter

# -- the category partition is the published one ------------------------------------------- #
assert Counter(C.marker_category_map().values()) == {
    "pre-syn": 12, "post-syn": 13, "dendrites": 4, "axons": 5}, "MERSCOPE partition changed"
assert len(C.marker_category_map()) == 34
assert C.marker_category_map()["Dlg4"] == "post-syn"
assert set(C.COMPARTMENT_OF) == set(CAT), "COMPARTMENT_OF does not cover the content categories"
if XENIUM_AVAILABLE:
    assert Counter(arms["Xenium"]["cat_map"].values()) == {
        "pre-syn": 10, "post-syn": 10, "dendrites": 2, "axons": 2}, "Xenium partition changed"
for name, a in arms.items():
    missing = [g for g in a["seed_genes"] if g not in a["cat_map"]]
    assert not missing, f"{name}: seeds without a category: {missing}"
print("[ok] marker partition and compartment map")

# -- complexity reproduces A2a -------------------------------------------------------------- #
ret_path = C.A2A_MULTIGENE_DIR / "retention_by_region.csv"
if ret_path.exists():
    ret = pd.read_csv(ret_path).query("brain_area == 'overall'")
    for s in C.SAMPLES:
        ds = C.dataset(s)
        r = ret.query("batch == @ds").iloc[0]
        mine = complexity.query("sample == @s and counting == 'nonNC'").iloc[0]
        assert int(mine["n_granules"]) == int(r["n_all"]), s
        assert int(mine["n_ge_3"]) == int(r["n_multigene"]), (s, mine["n_ge_3"], r["n_multigene"])
    print("[ok] frac_ge_3 reproduces a2a/multigene/retention_by_region.csv")
else:
    print("[skip] A2a retention table absent -- complexity cross-check not run")

# -- the 4x4 reconstructs from an independent per-granule loop ------------------------------ #
probe = np.random.default_rng(0).choice(M.shape[0], size=min(1000, M.shape[0]), replace=False)
for arm in content:
    ref = np.zeros((len(CAT), len(CAT)))
    for i in probe:
        si = CAT.index(seed_cat[i])
        for j, g in enumerate(marker_genes):
            if arm == "all_content" and g == combined["seed"][i]:
                continue
            if arm == "nonseed_content" and g in seedset:
                continue
            ref[si, CAT.index(cat_map[g])] += M[i, j]
    assert (ref == _table(seed_code[probe], content[arm][probe])).all(), (
        f"{arm}: the 4x4 does not reconstruct on the 1000-granule probe")
print("[ok] seed x content table reconstructs on a 1000-granule probe")

# -- the collapse is a partition of the 4x4, and the permutation null is centred on no effect - #
for arm, A in content.items():
    T = _table(seed_code, A)
    assert _collapse(T).sum() == T.sum(), f"{arm}: compartment collapse loses transcripts"
    assert np.allclose(null[arm]["fold4"].mean(axis=0), 1.0, atol=0.05), (
        f"{arm}: permutation null folds are not centred on 1 -- the shuffle is not doing what it "
        f"should:\n{null[arm]['fold4'].mean(axis=0)}")
print("[ok] compartment collapse conserves transcripts; permutation null centred on fold 1")

# -- subtype vocabulary --------------------------------------------------------------------- #
for name, a in arms.items():
    assert "axons" not in set(np.unique(a["subtype"])), (
        f"{name}: an 'axons' label appeared -- section 3 assumes none")
print("[ok] no pure axonal subtype in any arm")

In [ ]:
def _stamp(p):
    p = Path(p)
    return f"{p}@{int(p.stat().st_mtime)}" if p.exists() else f"{p}@MISSING"

run_info = pd.DataFrame([dict(
    xenium_included=XENIUM_AVAILABLE,
    n_perm=N_PERM, perm_seed=PERM_SEED, p_perm_floor=P_FLOOR,
    min_unique_genes=C.MIN_UNIQUE_GENES,
    complexity_levels=json.dumps(C.A2E_COMPLEXITY_LEVELS),
    exclude_nc_from_complexity=C.EXCLUDE_NC_FROM_COMPLEXITY,
    pure_subtypes=json.dumps(C.PURE_SUBTYPES),
    content_categories=json.dumps(CAT),
    compartment_of=json.dumps(C.COMPARTMENT_OF),
    content_arms=json.dumps(list(content)),
    n_markers_merscope=len(C.marker_category_map()),
    n_markers_xenium=len(arms["Xenium"]["cat_map"]) if XENIUM_AVAILABLE else None,
    n_granules_combined=int(combined["counts"].shape[0]),
    n_granules_per_arm=json.dumps({k: int(v["counts"].shape[0]) for k, v in arms.items()}),
    design_effect=json.dumps({a: round(float(nl["chi2"].mean() / (len(CAT) - 1) ** 2), 4)
                              for a, nl in null.items()}),
    combined_granule_adata=_stamp(C.COMBINED_GRANULE_ADATA),
    combined_subtype_labels=_stamp(C.COMBINED_SUBTYPE_LABELS),
    xenium_granule_adata=_stamp(C.XENIUM_GRANULE_ADATA) if XENIUM_AVAILABLE else None,
    xenium_subtype_labels=_stamp(C.XENIUM_SUBTYPE_LABELS) if XENIUM_AVAILABLE else None,
    elapsed_s=round(time.time() - T0, 1),
)])
run_info.to_csv(OUT / "run_info.csv", index=False)
print(run_info.T.to_string(header=False))
print("\nwrote:", ", ".join(sorted(p.name for p in OUT.glob("*.csv"))))